# BPE Tokenizer

Notebook for exploring and testing the byte-pair encoding tokenizer work for CS336 Assignment 1.

## Setup

Run this notebook from the repository root so local package imports and test paths resolve correctly.

In [10]:
from pathlib import Path

REPO_ROOT = Path.cwd()
REPO_ROOT

PosixPath('/home/ngejay3046/cs336-assignment-1')

## Data Inspection

Use this section to inspect small text samples before training or debugging the tokenizer.

In [11]:
sample_text = "low lower lowest wider widest"
sample_text

'low lower lowest wider widest'

## Tokenizer Experiments

Import your tokenizer implementation here once it is available through the assignment adapter or package modules.

In [12]:
# Example placeholder:
# from cs336_basics.tokenizer import BPETokenizer
# tokenizer = BPETokenizer(...)
# tokenizer.encode(sample_text)

## Focused Tests

Run tokenizer-specific tests from a terminal with:

```sh
uv run pytest tests/test_tokenizer.py
```

## Assignment Integration

### ord() and chr() Functions

In Python, you can use the ord() function to convert a single Unicode character into its integer representation. The chr() function converts an integer Unicode code point into a string with the corresponding character.

In [72]:
from IPython.display import display

display(ord('牛'))
display(chr(29275))

29275

'牛'

### Problem (unicode1): Understanding Unicode

(a) What Unicode character does `chr(0)` return?

(b) How does this character’s string representation differ from its printed representation?

(c) What happens when this character occurs in text?

**Answer.**

(a) `chr(0)` returns the Unicode character with code point `U+0000`, commonly called **NUL** or the **null character**. In Python's escaped representation, this appears as `'\x00'`.

(b) Its string representation shows an escaped form because the character itself is not visually printable. For example, `repr(chr(0))` returns `"'\\x00'"`. Printing it with `print(chr(0))` writes the actual null character to the output stream, but it appears invisible.

(c) When `U+0000` occurs in text, it usually has no visible glyph. Python allows it inside strings and treats it as an ordinary character of length 1. However, it can still cause issues in systems or formats that treat NUL specially. For example, older C-style string APIs historically use the null byte as a string terminator, so embedded NUL characters may affect how text is processed outside Python.

In [75]:
x = chr(0)
display(x)
display(repr(chr(0)))
print("chr(0):", x)
print("length:", len(x))
display("this is a test" + chr(0) + "string")
print("this is a test" + chr(0) + "string")

'\x00'

"'\\x00'"

chr(0):  
length: 1


'this is a test\x00string'

this is a test string


### Unicode Encodings

To encode a Unicode string into UTF-8, we can use the encode() function in Python. To access the underlying byte values for a Python bytes object, we can iterate over it (e.g., call list()). Finally, we can use the decode() function to decode a UTF-8 byte string into a Unicode string.

In [74]:
test_string = "hello! こんにちは!"

# Encode the string into UTF-8 bytes.
utf8_encoded = test_string.encode("utf-8")

# Display the encoded byte string.
print("Encoded byte string:", utf8_encoded)

# The type of the encoded string is 'bytes', which is a sequence of byte values (integers from 0 to 255).
print("Type of encoded byte string:", type(utf8_encoded))

# Get the list of byte values for the encoded string (a list of integers from 0 to 255).
print("Byte values:", list(utf8_encoded))

# One byte does not necessarily correspond to one Unicode character!
print("Length of original string:", len(test_string))

# The length of the UTF-8 encoded byte string is different from the length of the original Unicode string, 
# because some characters (like 'こ', 'ん', 'に', 'ち', 'は') are represented by multiple bytes in UTF-8.
print("Length of UTF-8 encoded byte string:", len(utf8_encoded))

# Finally, we can decode the UTF-8 byte string back into a Unicode string.
print("Decoded string:", utf8_encoded.decode("utf-8"))

Encoded byte string: b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'
Type of encoded byte string: <class 'bytes'>
Byte values: [104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]
Length of original string: 13
Length of UTF-8 encoded byte string: 23
Decoded string: hello! こんにちは!


### Problem (unicode2): Unicode Encodings

(a) What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than UTF-16 or UTF-32? It may be helpful to compare the output of these encodings for various input strings.

(b) Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string into a Unicode string. Why is this function incorrect? Provide an example of an input byte string that yields incorrect results.

```python
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))
```

> 'hello'

(c) Give a two-byte sequence that does not decode to any Unicode character(s).

**Answer.**

(a) UTF-8 is usually preferable for byte-level tokenizer training because it is compact for ASCII-heavy text, backwards-compatible with ASCII, and does not introduce many predictable zero bytes. For example, the ASCII character `a` is one byte in UTF-8 (`61`), but two bytes in UTF-16 (`61 00` in little-endian form) and four bytes in UTF-32 (`61 00 00 00`). Since much web and code text is ASCII-heavy, UTF-16 and UTF-32 would make the training corpus longer and force the tokenizer to model many artificial bytes that come from the encoding rather than from the text itself. UTF-8 also has no required byte-order mark and is the standard encoding for most modern text data, so training on UTF-8 bytes better matches the data distribution we usually want to tokenize.

(b) The function is incorrect because UTF-8 characters may require multiple bytes. It decodes each byte independently, so it only works for single-byte ASCII characters. For a multibyte character, the individual bytes are not valid standalone UTF-8 strings. For example, `"é".encode("utf-8")` is `b'\xc3\xa9'`; decoding `b'\xc3'` by itself raises a `UnicodeDecodeError` because it is only the first byte of a two-byte UTF-8 sequence. Similarly, `"こんにちは".encode("utf-8")` contains three-byte UTF-8 sequences, and decoding one byte at a time fails.

(c) One example is `b'\xc3\x28'`. The byte `0xc3` indicates the start of a two-byte UTF-8 sequence, but `0x28` is not a valid continuation byte because UTF-8 continuation bytes must be in the range `0x80` to `0xbf`.


### BPE Tokenizer Training

The BPE tokenizer training procedure consists of three main steps.

#### Vocabulary initialization

The tokenizer vocabulary is a one-to-one mapping from bytestring token to integer ID. Since we’re training a byte-level BPE tokenizer, our initial vocabulary is simply the set of all bytes. Since there are 256 possible byte values, our initial vocabulary is of size 256.

#### Pre-tokenization

Most modern tokenizers use a regex-based pre-tokenizer, a practice from GPT-2. We’ll use a slightly prettier form of the original regex, fetched from 
[github.com/openai/tiktoken/pull/234/files](https://github.com/openai/tiktoken/pull/234/files):

```python
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
```

Use re.finditer to avoid storing the pre-tokenized words as you construct your mapping from pre-tokens to their counts. For example: 

```python
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
import regex as re
def get_pre_tokens(text: str):
    return [m.group(0) for m in re.finditer(PAT, text)]

pre_tokens = get_pre_tokens("sample_text")
print(pre_tokens)
```

> ['sample', '_', 'text']



